# Autenticación y autorización: explicación paso a paso

Autenticar significa comprobar quién hace una petición. Autorizar significa comprobar qué puede consultar esa identidad. Que el bot tenga un token válido no permite mostrarle cualquier reparación a cualquier cliente.



## Alineación con los requisitos

Los tres PDF originales ya fueron revisados. Consulta [el plan integrado Bot, Backend y Frontend](PLAN_CHATBOT.ipynb). El prototipo PostgreSQL directo mostrado en esta guía requiere adaptación: los documentos establecen acceso a la base mediante Backend. FAQ y periodos serán administrables; no se incluye editor de comportamientos nuevos.


## Qué significa cada credencial

| Dato | Quién lo usa | Qué demuestra | Qué no demuestra |
| --- | --- | --- | --- |
| `META_ACCESS_TOKEN` | Bot ante Meta | Permiso para usar la API de WhatsApp de la cuenta | Que un cliente sea dueño de una orden |
| `META_PHONE_NUMBER_ID` | Bot ante Meta | Identifica el número empresarial | No es una contraseña ni el teléfono del cliente |
| `META_VERIFY_TOKEN` | Meta y webhook durante su alta GET | Que el endpoint responde al reto configurado | No valida la firma de cada mensaje POST |
| Usuario/contraseña PostgreSQL | Bot ante PostgreSQL | Permiso de conexión a la base de conversaciones | No verifica al cliente de WhatsApp |
| `BACKEND_BOT_TOKEN` | Bot ante Backend | Identidad del servicio chatbot | No concede automáticamente acceso a una orden ajena |
| `ctx.from` | Flujo y Backend | Remitente entregado por el proveedor | Por sí solo no prueba propiedad sin verificar webhook y vinculación previa |
| Folio | Cliente y Backend | Identifica el servicio solicitado | No es una contraseña ni autorización |

El nombre interno `jwtToken` del proveedor recibe `META_ACCESS_TOKEN`: no implica que debas generar un JWT propio para Meta. El token del Backend es otra credencial; no reutilices el de Meta. El contrato propone Bearer, pero el Backend aún debe implementar su validación y permisos.



## Ejemplo concreto

Ana tiene la orden FIX-001 y un teléfono previamente verificado en recepción. Bruno conoce ese folio. Ambos escriben al bot. El token del bot será el mismo en las dos peticiones, porque identifica a la aplicación. Lo que cambia es el remitente `ctx.from`. El Backend debe comprobar que el remitente corresponde al propietario de FIX-001; sólo Ana recibe el estado. Bruno recibe el mismo mensaje que recibiría con un folio inexistente.

No pidas al cliente que escriba su teléfono como prueba de identidad: podría escribir el de otra persona. No envíes contraseñas del personal por WhatsApp. Vincular un teléfono previamente verificado es una propuesta del ejemplo, pendiente de aprobación funcional con el equipo; si se requiere OTP, debes implementar generación, caducidad, límites de intentos y validación en Backend. El ejemplo actual no implementa OTP.



## Secuencia de consulta autorizada

```mermaid
sequenceDiagram
    actor Cliente
    participant Meta
    participant Bot
    participant Backend
    participant Negocio as PostgreSQL negocio
    Cliente->>Meta: estado y luego folio
    Meta->>Bot: webhook con remitente
    Note over Meta,Bot: Verificar autenticidad del POST antes de confiar en remitente
    Bot->>Bot: validar formato del folio
    Bot->>Backend: POST folio + ctx.from + Bearer token del bot
    Backend->>Backend: autenticar token del bot
    Backend->>Negocio: buscar orden y propietario
    Backend->>Backend: comprobar teléfono verificado y propiedad
    alt Propietario autorizado
        Backend-->>Bot: 200 con campos públicos
        Bot-->>Meta: estado público
        Meta-->>Cliente: respuesta
    else Folio ajeno o inexistente
        Backend-->>Bot: 403 o 404
        Bot-->>Meta: mismo mensaje genérico
        Meta-->>Cliente: no se pudo verificar
    else Credencial del bot inválida
        Backend-->>Bot: 401
        Bot-->>Meta: consulta no disponible
    end
```



## Qué hace hoy el código y qué falta

El flujo toma `ctx.from` y el cliente HTTP lo envía con el folio. El cliente HTTP agrega `Authorization: Bearer ...`, selecciona campos públicos y convierte errores en respuestas controladas. En demo sólo se compara con `DEMO_PHONE_NUMBER`: es un dato ficticio para practicar, no autenticación productiva.

El endpoint del contrato todavía no existe. Faltan autenticación del bot, autorización por propietario, normalización acordada del teléfono, límites de intentos y auditoría en Backend. El código inspeccionado de MetaProvider 1.4.2 registra el reto GET y el receptor POST; no se debe dar por implementada la validación de firma del POST. Antes de activar consultas reales, verificar la firma de Meta con el secreto de la aplicación sobre el cuerpo original en un middleware compatible o un gateway. Esa implementación queda pendiente; `META_VERIFY_TOKEN` no la reemplaza.



## Dónde se guardan los datos

```mermaid
flowchart LR
    W[WhatsApp cliente] --> M[Meta]
    M --> F[Flujo BuilderBot]
    F --> H[(PostgreSQL chatbot: history y contact)]
    F --> A[Backend: autentica y autoriza]
    A --> N[(PostgreSQL negocio: clientes y órdenes)]
```

El adaptador persiste respuestas y eventos con teléfonos. El historial contiene información del cliente y exige control de acceso y retención. Las tablas del adaptador no son las tablas `clientes` y `ordenes_servicio`. Usa `fixify_chatbot` como base dedicada para evitar colisiones y permisos sobre el negocio. No escribas tokens en mensajes ni en los datos guardados por el flujo.

Fuentes: [adaptadores de BuilderBot](https://www.builderbot.app/en/databases), [proveedor Meta](https://www.builderbot.app/en/providers/meta). Diagramas en celdas Markdown con bloques Mermaid; utiliza un visor de notebooks que soporte Mermaid, como JupyterLab moderno. Si el visor sólo muestra texto, abre el notebook en JupyterLab o habilita soporte Mermaid.


## Validación de esta versión

Se verificó con PostgreSQL temporal, puerto 55439 y datos ficticios: creación de tablas/funciones y recuperación de un evento y contacto desde una segunda conexión. Pasaron cinco pruebas, ESLint, compilación y revisión de tipos. La instancia temporal se cerró al terminar. No se modificó la base del negocio ni se probó el webhook real de Meta.

`pnpm test` ejecuta las pruebas de consulta y omite la integración PostgreSQL por defecto. Para ejecutar integración usa una base dedicada de pruebas con `CHATBOT_TEST_POSTGRES=1`, `PG_TEST_HOST`, `PG_TEST_PORT`, `PG_TEST_USER`, `PG_TEST_PASSWORD` y `PG_TEST_DATABASE`. El test crea tablas y funciones del adaptador y elimina sus propios registros al terminar; no lo ejecutes contra la base productiva.
